# 13 · The performer's measurements from the front

Notebook 12 holds the body to what the config knows of the performer (`performer.measurements`,
`performer.phenotype`, `fitting.body_height`). For a reference video with none of that, this
notebook estimates them from a few frames that show her from the front, and writes them where
notebook 12 can take them as its prior: height, weight, bust, underbust, waist, hips, the feet
(and the EU shoe size), the breasts' cupsize and firmness.

| Step | What |
| --- | --- |
| Frames | the `FRAMES` frames of notebook 12's prepared pieces that face the camera most squarely, whole and upright, `MIN_GAP` seconds apart |
| Surface | Sapiens2's keypoints, normals and pointmap and Depth Pro's depth on them (`preprocess.surface`), and the field of view |
| Body | one Anny shape for all of them at a nominal height, a pose each: the motion fit poses, the shape fit reshapes, in turns (`fitting.performer`) |
| Scale | how much larger she is than the fit: the iris, the distance between the pupils, Depth Pro's metric depth |
| Result | the fitted mesh measured (`body_models.measurements`) and scaled: `outputs/video_avatar/<name>/performer.yaml` |

**What the frames can tell.** The outline from the front gives widths, not depths: the
estimated normals and depth give the breasts' and belly's relief, which sets cupsize, firmness
and the circumferences. One camera cannot tell size from distance, so the height comes from
things of nearly the same size in every adult (iris 11.7 ± 0.5 mm, pupils 62 ± 3.5 mm apart)
and from Depth Pro's metric depth: each a few per cent off at best, and the iris needs a face
of more than a few dozen pixels. The feet are seen end-on from the front: their length follows
mostly from Anny's proportions, so the shoe size is the weakest estimate.

Needs notebook 12's parts 1 and 2 run on the same `VIDEO` (its prepared pieces).

In [ ]:
import dataclasses
import logging
from pathlib import Path

import av
import cv2
import matplotlib.pyplot as plt
import numpy as np
import nvdiffrast.torch as dr
import torch
import torch.nn.functional as F
import yaml

from avatar_splats.body_models.anny import AnnyBody
from avatar_splats.body_models.measurements import BodyMeasurements
from avatar_splats.body_models.skinning import linear_blend_skinning, sparsify_weights
from avatar_splats.cameras import Camera
from avatar_splats.config import MEASUREMENT_NAMES, load_config
from avatar_splats.data.video import read_frames
from avatar_splats.fitting.body import BONE_INFLUENCES, MIN_KEYPOINT_CONFIDENCE
from avatar_splats.fitting.performer import (
    FramesShape,
    FrameTargets,
    ShapeFitSettings,
    anny_height,
    eu_shoe_size,
    fit_frames_shape,
    frontal_scores,
    iris_diameters,
    resized,
    scale_votes,
    scaled_measurements,
)
from avatar_splats.fitting.silhouette import (
    SurfaceSamples,
    overlay,
    render_silhouette,
)
from avatar_splats.motion.fit import fit_motion
from avatar_splats.motion.observe import Observations, mask_size
from avatar_splats.preprocess.keypoints import (
    coco_keypoints,
    estimate_sapiens_keypoints,
    extra_keypoints,
    hand_landmarks,
)
from avatar_splats.preprocess.prepare import (
    NOT_THEM,
    PART_NAMES,
    BodyParts,
    PreparedShot,
    skin_coloured,
)
from avatar_splats.preprocess.surface import SurfaceSettings, estimate_surfaces, lift_keypoints
from avatar_splats.rendering.surface import render_surface

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
CONFIG = (
    "../configs/default.yaml"  # the fitting and motion settings; its performer is only compared
)
config = load_config(CONFIG)
device = "cuda"

## Settings

`VIDEO` is notebook 12's. Everything the config says of the performer is estimated anew except
`HELD_PHENOTYPE`; where the config does give a value, it is shown beside the estimate. Set
`HEIGHT` where her height is known: the votes are then only compared with it.

In [ ]:
VIDEO = Path("../examples/live-performance/water.mp4")  # relative to notebooks/; notebook 12's
NAME = VIDEO.stem

FRAMES = 6  # frames fitted: more see more poses, each costs a motion fit
MIN_GAP = 2.0  # seconds between the frames chosen, so they are not one pose
MIN_FRONTAL = 0.5  # frames facing the camera less squarely (frontal_scores) are not chosen
MAX_OVERLAP = 0.2  # frames whose box another person's covers more than this are not chosen
BODY_PARTS_MODEL = "facebook/sapiens2-seg-1b"  # her outline, hair and clothes on the chosen frames
IGNORED_PARTS = (
    "Hair", "Apparel", "Upper_Clothing", "Lower_Clothing", "Left_Shoe", "Right_Shoe",
    "Left_Sock", "Right_Sock", "Eyeglass",
)  # fmt: skip
SKIN_DISTANCE = 3.0  # clothing this close to the skin's colour is skin Sapiens2 mislabelled
IGNORE_MARGIN = 4  # pixels the ignored parts are widened by
EDGE_BAND = 4  # pixels off the outline's edge where normals and depth are not compared

POSE_MODEL = "facebook/sapiens2-pose-1b"  # keypoints of the chosen frames, over ViTPose's
SURFACE = SurfaceSettings(
    normal_model="facebook/sapiens2-normal-1b",
    pointmap_model="facebook/sapiens2-pointmap-1b",
    depth_model="apple/DepthPro-hf",
    normal_size=(768, 576),
    batch=4,
    depth_batch=2,
    person_margin=0.08,
    map_side=512,
)
FOV_Y_DEGREES = None  # degrees; None: the median of Depth Pro's and Sapiens2's

# The body is fitted this tall; the scale votes say how much larger she is.
NOMINAL_HEIGHT = 1.65
HEIGHT = None  # metres, where known: the votes are then only shown
HEIGHT_VOTES = ("iris", "pupils", "Depth Pro")  # which count, by their median
MIN_IRIS_PIXELS = 6.0  # frames with smaller irises give the iris and pupil votes nothing

# Held as given: what the frames cannot tell. Age is held too: the outline alone would trade it
# for proportions.
HELD_PHENOTYPE = {"gender": config.performer.phenotype.get("gender", 1.0), "age": 0.5}
ROUNDS = 2  # rounds of motion fit and shape fit
MOTION_ITERATIONS = [400, 400, 300]
SHAPE = ShapeFitSettings(
    iterations=400,
    shape_learning_rate=0.01,
    pose_learning_rate=0.002,
    weights={
        "keypoints": config.fitting.keypoint_weight,
        "silhouette": config.fitting.silhouette_weight,
        "normals": 0.2,  # per 1 - cosine: the breasts' and belly's relief
        "depth": 0.5,  # per metre, the estimate scaled to agree in median
        "height": config.fitting.height_weight,
        "phenotype_prior": 0.05,  # per squared step from Anny's average: few frames, few pixels
        "local_changes": config.fitting.local_change_weight,
        "pose_prior": 1.0,
        "joint_limits": config.motion.joint_limit_weight,
    },
    held=tuple(HELD_PHENOTYPE),
    surface_samples=config.motion.surface_samples,
    silhouette_inset=config.fitting.silhouette_inset,
)
NEAR, FAR = 0.1, 20.0  # metres

out_dir = config.outputs / "video_avatar" / NAME
piece_paths = sorted((out_dir / "prepared").glob("piece-*.npz"))
if not piece_paths:
    raise FileNotFoundError(f"no prepared pieces in {out_dir}: run notebook 12's parts 1 and 2")
print(f"{len(piece_paths)} prepared pieces of {VIDEO.name}")

## The frames

Every usable frame of every piece scored by how squarely it faces the camera (all 17 keypoints
placed, her left on the image's right, shoulders and hips as wide as from the front, the torso
upright), times its sharpness against the median (at most 1); frames where another person's box
covers more than `MAX_OVERLAP` of hers are left out. The best `FRAMES` at least `MIN_GAP` seconds
apart, then Sapiens2's body parts on them: her outline, and the hair and clothes that are not
compared.

In [ ]:
fps = None
candidates = []  # (score, piece, frame in piece, video frame)
sharpness = []
for p, path in enumerate(piece_paths):
    stored = np.load(path)
    usable = np.flatnonzero(stored["usable"])
    frontal = frontal_scores(stored["keypoints"][usable, :17], MIN_KEYPOINT_CONFIDENCE)
    alone = np.nan_to_num(stored["overlap"][usable], nan=1.0) <= MAX_OVERLAP
    sharpness.append(stored["sharpness"][usable])
    candidates += [
        (f, p, int(i), int(stored["frame_indices"][i]), float(sh))
        for f, ok, i, sh in zip(frontal, alone, usable, stored["sharpness"][usable], strict=True)
        if f >= MIN_FRONTAL and ok
    ]
median_sharpness = np.nanmedian(np.concatenate(sharpness))
candidates = sorted(
    ((score * min(1.0, np.nan_to_num(sh / median_sharpness)), p, i, v)
     for score, p, i, v, sh in candidates),
    reverse=True,
)  # fmt: skip
with av.open(str(VIDEO)) as container:
    fps = float(container.streams.video[0].average_rate)
chosen = []
for candidate in candidates:
    if all(abs(candidate[3] - c[3]) >= MIN_GAP * fps for c in chosen):
        chosen.append(candidate)
    if len(chosen) == FRAMES:
        break
if not chosen:
    raise LookupError("no frame faces the camera: lower MIN_FRONTAL")
chosen.sort(key=lambda c: c[3])
print(f"{len(candidates)} frames face the camera; chosen:")
for score, p, _, v in chosen:
    print(f"  piece {p}, video frame {v} ({v / fps:.1f} s): score {score:.2f}")

columns = {k: [] for k in ("frames", "boxes", "keypoints", "face")}
for _, p, i, v in chosen:
    piece = PreparedShot.load(piece_paths[p])
    width_, height_ = (int(v) for v in piece.image_size)
    frame = next(read_frames(VIDEO, max_frames=1, start=v))
    columns["frames"].append(cv2.resize(frame, (width_, height_), interpolation=cv2.INTER_AREA))
    columns["boxes"].append(piece.boxes[i])
    columns["keypoints"].append(piece.keypoints[i, :17])
    columns["face"].append(piece.face_landmarks[i])
frames, boxes, keypoints, face_landmarks = (
    np.stack(columns[k]) for k in ("frames", "boxes", "keypoints", "face")
)
N, (HEIGHT_PX, WIDTH_PX) = len(frames), frames.shape[1:3]
del columns
body_parts, _ = BodyParts(BODY_PARTS_MODEL, FRAMES, SURFACE.person_margin, device)(frames, boxes)
masks = body_parts != NOT_THEM
torch.cuda.empty_cache()

# Hair and clothes are not compared; clothing coloured like the skin is skin.
labelled = np.isin(body_parts, [PART_NAMES.index(n) for n in IGNORED_PARTS])
clothing = labelled & (body_parts != PART_NAMES.index("Hair"))
freed = np.stack(
    [
        skin_coloured(f, b, c, SKIN_DISTANCE)
        for f, b, c in zip(frames, body_parts, clothing, strict=True)
    ]
)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * IGNORE_MARGIN + 1,) * 2)
ignored = np.stack(
    [
        cv2.dilate((a & ~b).astype(np.uint8), kernel) > 0
        for a, b in zip(labelled, freed, strict=True)
    ]
)

fig, axes = plt.subplots(1, N, figsize=(18, 18 / N * HEIGHT_PX / WIDTH_PX), squeeze=False)
for ax, k in zip(axes[0], range(N), strict=True):
    picture = frames[k].copy()
    picture[ignored[k]] = (0.4 * picture[ignored[k]] + [150, 0, 150]).astype(np.uint8)
    picture[~masks[k]] //= 3
    ax.imshow(picture)
    ax.set_title(f"frame {chosen[k][3]}: {chosen[k][0]:.2f}", fontsize=9)
    ax.axis("off")
fig.suptitle("the chosen frames; ignored (hair, clothes) in magenta")
plt.show()

## The surface and the field of view

Sapiens2's keypoints (over ViTPose's), its normals and pointmap, and Depth Pro's depth, on the
chosen frames. Their focal lengths are two estimates of the field of view.

In [ ]:
goliath = estimate_sapiens_keypoints(frames, boxes, POSE_MODEL, device)
keypoints, extra, hands = coco_keypoints(goliath), extra_keypoints(goliath), hand_landmarks(goliath)
keys = [f"frame-{c[3]}" for c in chosen]
surface = estimate_surfaces(frames, masks, boxes, keys, SURFACE, device)
torch.cuda.empty_cache()
MAP_WIDTH, MAP_HEIGHT = (int(v) for v in surface.map_size)


def fov_of(focal):
    return float(np.degrees(2 * np.arctan(HEIGHT_PX / 2 / np.nanmedian(focal))))


fov_estimates = {"Depth Pro": fov_of(surface.depth_focal), "Sapiens2": fov_of(surface.point_focal)}
if FOV_Y_DEGREES is None:
    FOV_Y_DEGREES = round(float(np.median(list(fov_estimates.values()))), 1)
print(", ".join(f"{k} {v:.1f}°" for k, v in fov_estimates.items()), f"-> {FOV_Y_DEGREES:g}°")


def normal_colours(normals):
    # (..., 3) camera-axes normals as RGB: red right, green up, blue towards the camera
    return ((normals * np.array([1, -1, -1]) + 1) / 2).clip(0, 1) * (
        np.abs(normals).sum(-1, keepdims=True) > 0
    )


fig, axes = plt.subplots(2, N, figsize=(18, 2 * 18 / N * HEIGHT_PX / WIDTH_PX), squeeze=False)
for k in range(N):
    axes[0, k].imshow(frames[k])
    axes[1, k].imshow(normal_colours(surface.normals[k].astype(np.float32)))
for ax in axes.flat:
    ax.axis("off")
plt.show()

## The body

Each frame is its own "person" for the motion fit (`independent`: one step for all), three
copies long, so its smoothness terms see a still pose. Then the shape fit: one phenotype and
one set of local changes for all frames, `NOMINAL_HEIGHT` tall, the poses corrected a little,
against keypoints, outline, normals and depth. `ROUNDS` times; Anny starts at her average.

In [ ]:
body = AnnyBody().to(device)
COPIES = 3
mask_width, mask_height = mask_size((WIDTH_PX, HEIGHT_PX), config.motion.resolution)
motion_config = dataclasses.replace(
    config.motion,
    fov_y_degrees=FOV_Y_DEGREES,
    iterations=MOTION_ITERATIONS,
    normal_weight=0.1,
    depth_weight=0.5,
    keypoint_depth_weight=1.0,
)


def tensor(array):
    return torch.from_numpy(np.asarray(array)).float().to(device)


def copies(values):
    # (N, ...) -> (N, COPIES, ...): each frame a still "person"
    return np.repeat(np.asarray(values)[:, None], COPIES, axis=1)


def packed(flags):
    small = F.interpolate(tensor(flags)[:, None], (mask_height, mask_width), mode="area")[:, 0]
    return copies(np.packbits((small > 0.5).cpu().numpy(), axis=-1))


mask_normals = resized(torch.from_numpy(surface.normals).float(), (mask_height, mask_width))
mask_normals = F.normalize(mask_normals, dim=-1) * (mask_normals.norm(dim=-1, keepdim=True) > 0.5)
mask_depths = torch.stack(
    [
        resized(torch.from_numpy(d).float(), (mask_height, mask_width))
        for d in (surface.point_depth, surface.depth)
    ],
    dim=1,
)
observations = Observations(
    frame_indices=np.arange(COPIES),
    fps=fps,
    image_size=np.array([WIDTH_PX, HEIGHT_PX]),
    mask_size=np.array([mask_width, mask_height]),
    masks=packed(masks),
    boxes=copies(boxes),
    keypoints=copies(keypoints),
    hands=copies(hands),
    blendshapes=np.zeros((N, COPIES, 0), np.float32),
    blendshape_names=np.array([], str),
    ignored=packed(ignored),
    keypoint_depths=copies(
        lift_keypoints(
            keypoints, surface.point_depth, (WIDTH_PX, HEIGHT_PX), MIN_KEYPOINT_CONFIDENCE
        )
    ),
    extra_keypoints=copies(extra),
    normals=copies(mask_normals.numpy().astype(np.float16)),
    depths=copies(mask_depths.numpy().astype(np.float16)),
)

# The shape fit's targets, at the fitting resolution.
scale = config.fitting.resolution / max(WIDTH_PX, HEIGHT_PX)
frame_camera = Camera.fixed(WIDTH_PX, HEIGHT_PX, FOV_Y_DEGREES).to(device)
camera = frame_camera.scaled(scale)
size = (camera.height, camera.width)
target_masks = resized(tensor(masks), size, "area")
compared = (resized(tensor(ignored), size, "area") < 0.5).float()
hard = (target_masks > 0.5).float()[:, None]
band = max(1, EDGE_BAND // 2)
inner = -F.max_pool2d(-hard, 2 * band + 1, stride=1, padding=band)[:, 0]
normals_ = resized(torch.from_numpy(surface.normals).float().to(device), size)
has_normal = normals_.norm(dim=-1, keepdim=True) > 0.5


def confident(points):
    points = tensor(np.nan_to_num(points))
    points[..., :2] *= scale
    points[..., 2] *= points[..., 2] >= MIN_KEYPOINT_CONFIDENCE
    return points


targets = FrameTargets(
    masks=target_masks,
    compared=compared,
    keypoints=confident(keypoints),
    extra_keypoints=confident(extra),
    normals=F.normalize(normals_, dim=-1) * has_normal,
    depths=torch.stack(
        [resized(tensor(d.astype(np.float32)), size) for d in (surface.point_depth, surface.depth)],
        1,
    ),
    surface_region=inner * compared * has_normal[..., 0],
)

start_phenotype = body.default_phenotype()[0].cpu().numpy()
for label, value in HELD_PHENOTYPE.items():
    start_phenotype[body.phenotype_labels.index(label)] = value
shape = FramesShape(
    phenotype=start_phenotype,
    local_changes=np.zeros(len(body.local_change_labels), np.float32),
    rotations=np.zeros((N, body.num_bones, 3), np.float32),
    translation=np.zeros((N, 3), np.float32),
)
for round_ in range(ROUNDS):
    motion = fit_motion(
        observations, body, shape, [], motion_config, config.fitting.silhouette_inset,
        independent=True,
    )  # fmt: skip
    print(
        f"round {round_ + 1}: motion fit IoU {motion.silhouette_iou[:, 0].mean():.3f}, "
        f"keypoints {motion.keypoint_error[:, 0].mean():.1f} px"
    )
    shape = dataclasses.replace(
        shape, rotations=motion.rotations[:, 0], translation=motion.translation[:, 0]
    )
    shape = fit_frames_shape(body, shape, targets, camera, NOMINAL_HEIGHT, SHAPE, NEAR, FAR)
    print(
        f"round {round_ + 1}: shape fit IoU {shape.silhouette_iou.mean():.3f}, "
        f"keypoints {shape.keypoint_error.mean() / scale:.1f} px"
    )
print(dict(zip(body.phenotype_labels, shape.phenotype.round(2), strict=True)))

The fit per frame: its outline over hers (white both, red only hers, cyan only the body's;
ignored pixels grey), then the estimated normals over the fitted body's.

In [ ]:
with torch.no_grad():
    out = body(
        tensor(shape.phenotype)[None],
        tensor(shape.local_changes)[None],
        tensor(shape.rotations),
        tensor(shape.translation),
    )
    vertices = linear_blend_skinning(
        out["rest_vertices"], *sparsify_weights(body.skin_weights, BONE_INFLUENCES), out["skinning"]
    )
    samples = SurfaceSamples.draw(
        out["rest_vertices"][0], body.faces, config.motion.surface_samples
    )
    silhouettes = (
        render_silhouette(
            samples.points(vertices, body.faces, config.fitting.silhouette_inset * samples.radius),
            samples.radius,
            camera,
        )
        .cpu()
        .numpy()
    )
    drawn = render_surface(dr.RasterizeCudaContext(), vertices, body.faces, camera, NEAR, FAR)
fig, axes = plt.subplots(3, N, figsize=(18, 3 * 18 / N * HEIGHT_PX / WIDTH_PX), squeeze=False)
for k in range(N):
    image = overlay(target_masks[k].cpu().numpy(), silhouettes[k])
    image[compared[k].cpu().numpy() < 0.5] = 90
    axes[0, k].imshow(image)
    axes[0, k].set_title(
        f"IoU {shape.silhouette_iou[k]:.3f}, {shape.keypoint_error[k] / scale:.0f} px", fontsize=9
    )
    axes[1, k].imshow(normal_colours(targets.normals[k].cpu().numpy()))
    axes[2, k].imshow(
        normal_colours((drawn.normals[k] * drawn.covered[k, ..., None]).cpu().numpy())
    )
for ax in axes.flat:
    ax.axis("off")
for row, label in enumerate(("outline", "estimated normals", "fitted normals")):
    axes[row, 0].set_title(label, loc="left", fontsize=9)
plt.show()

## Her height

How many times larger she is than the fit, per frame and anchor: the iris's diameter and the
distance between the pupils (MediaPipe's landmarks, at the depth of the fitted body's eyes; only
where the iris spans `MIN_IRIS_PIXELS`), Depth Pro's metric depth of her against the fitted
body's (its median ratio). Her height is the median of the `HEIGHT_VOTES`' medians, or `HEIGHT` where given.

In [ ]:
person = resized(tensor(masks & ~ignored), (MAP_HEIGHT, MAP_WIDTH), "area") > 0.5
votes = scale_votes(
    body,
    shape,
    frame_camera,
    face_landmarks,
    tensor(surface.depth.astype(np.float32)),
    person,
    NEAR,
    FAR,
)
small_iris = ~(np.nan_to_num(iris_diameters(face_landmarks)) >= MIN_IRIS_PIXELS)
votes["iris"][small_iris] = np.nan
votes["pupils"][small_iris] = np.nan
heights = {name: NOMINAL_HEIGHT * v for name, v in votes.items()}
medians = {
    name: float(np.nanmedian(h)) if np.isfinite(h).any() else np.nan for name, h in heights.items()
}
counted = [medians[n] for n in HEIGHT_VOTES if np.isfinite(medians.get(n, np.nan))]
estimated_height = float(np.median(counted)) if counted else np.nan
height = HEIGHT if HEIGHT is not None else estimated_height
if not np.isfinite(height):
    raise ValueError("no vote could tell her height: set HEIGHT")
known_height = config.fitting.body_height
for name, h in heights.items():
    per_frame = " ".join("  -  " if not np.isfinite(v) else f"{v:.2f}" for v in h)
    print(f"{name:>12}: {medians[name]:.3f} m   per frame {per_frame}")
print(f"estimated {estimated_height:.3f} m; used {height:.3f} m (config: {known_height} m)")

fig, ax = plt.subplots(figsize=(10, 3))
for row, (name, h) in enumerate(heights.items()):
    ax.scatter(h, np.full(len(h), row), s=20)
    ax.plot([medians[name]] * 2, [row - 0.3, row + 0.3], color="k")
ax.axvline(height, color="tab:green", label="used")
ax.axvline(known_height, color="grey", ls="--", label="config")
ax.set_yticks(range(len(heights)), list(heights))
ax.set_xlabel("height (m)")
ax.legend()
plt.show()

## Her measurements

The fitted rest mesh measured as a tailor would (`BodyMeasurements`: the circumferences round
the torso's slices, the arms left out; the weight from the volume) and scaled to her height:
lengths by the scale, the weight by its cube. Cupsize and firmness are the fitted phenotype's,
whatever her size. Then her rest mesh from the front and the side.

In [ ]:
factor = height / anny_height(body, shape.phenotype, shape.local_changes)
with torch.no_grad():
    zeros = torch.zeros(1, body.num_bones, 3, device=device)
    rest = body(
        tensor(shape.phenotype)[None], tensor(shape.local_changes)[None], zeros, zeros[:, 0]
    )["rest_vertices"][0]
    measured = {k: float(v) for k, v in BodyMeasurements(body)(rest).items()}
measured = scaled_measurements(measured, factor)
phenotype = dict(zip(body.phenotype_labels, shape.phenotype.astype(float), strict=True))
known = {**config.performer.measurements, **config.performer.phenotype}
units = {"weight": "kg"}
print(f"{'':>10} {'estimate':>10} {'config':>10}")
print(f"{'height':>10} {height:10.3f} {known_height:10.3f}")
nan = float("nan")
for name in MEASUREMENT_NAMES:
    print(f"{name:>10} {measured[name]:10.3f} {known.get(name, nan):10.3f} {units.get(name, 'm')}")
shoe = eu_shoe_size(measured["foot"])
known_shoe = eu_shoe_size(known["foot"]) if "foot" in known else nan
print(f"{'shoe (EU)':>10} {shoe:10.1f} {known_shoe:10.1f}")
for name in ("cupsize", "firmness"):
    print(f"{name:>10} {phenotype[name]:10.2f} {known.get(name, float('nan')):10.2f}")

# Front and side, at her size.
view = Camera.fixed(512, 1024, 30.0).to(device)
scaled_rest = rest * factor
middle = (scaled_rest.max(0).values + scaled_rest.min(0).values) / 2
distance = 1.1 * height / (2 * np.tan(np.radians(15)))
fig, axes = plt.subplots(1, 2, figsize=(8, 8))
sides = (("front", (0.0, -1.0, 0.0)), ("side", (1.0, 0.0, 0.0)))
for ax, (label, direction) in zip(axes, sides, strict=True):
    eye = middle + distance * torch.tensor(direction, device=device)
    seen = view.looking_at(eye, middle)
    with torch.no_grad():
        image = render_surface(
            dr.RasterizeCudaContext(), scaled_rest[None], body.faces, seen, NEAR, FAR
        )
    shade = (-image.normals[0, ..., 2]).clamp(0, 1) * image.covered[0]
    ax.imshow(shade.cpu().numpy(), cmap="gray")
    ax.set_title(label)
    ax.axis("off")
plt.show()

## For notebook 12

Written to `outputs/video_avatar/<name>/performer.yaml`, in the config's own sections: paste
`performer` and `fitting.body_height` into the performer's config (cupsize and firmness go in
`phenotype`, held through the fit), or keep only what looks right. Rounded as a tape would read.

In [ ]:
estimate = {
    "performer": {
        "measurements": {
            name: float(round(measured[name], 1 if name == "weight" else 3))
            for name in MEASUREMENT_NAMES
        },
        "phenotype": {
            **{k: float(v) for k, v in HELD_PHENOTYPE.items() if k != "age"},
            "cupsize": float(round(phenotype["cupsize"], 2)),
            "firmness": float(round(phenotype["firmness"], 2)),
        },
    },
    "fitting": {"body_height": float(round(height, 3))},
}
path = out_dir / "performer.yaml"
path.write_text(
    f"# estimated by notebook 13 from {len(chosen)} frames of {VIDEO.name}; "
    f"EU shoe size {shoe:.1f}\n" + yaml.safe_dump(estimate, sort_keys=False)
)
print(path.read_text())